# Day 14 · LoRA / QLoRA 原理与实现

**配套讲义**: [`days/day-14.md`](../days/day-14.md) ｜ **需要 GPU（云机器）**

手写一个最小 LoRA 层（低秩 A/B 矩阵），验证「B 零初始化 → 输出与基座逐元素相等」，并搞清 `r` / `alpha` / `target_modules` 在 VLM 里怎么选。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w3.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys, torch
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.version.cuda, "| available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"gpu    : {p.name}  {p.total_memory / 1024**3:.0f} GB")
    print("bf16   :", torch.cuda.is_bf16_supported())
else:
    print("⚠️  没有 GPU —— 这一天的训练/推理跑不了。先看 docs/13-hardware-and-cost.md 租机器")

## 1. 十行手写一遍 LoRA，确认你真的懂

不看 `lora_utils.py`，自己写一遍最小版本。核心只有一行：
`y = Wx + (alpha/r) · B(Ax)`，其中 **B 初始化为零**。

In [ ]:
import torch, torch.nn as nn

class MiniLoRA(nn.Module):
    def __init__(self, base: nn.Linear, r=16, alpha=32):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False          # 原权重冻结
        d_in, d_out = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, d_in)); nn.init.kaiming_uniform_(self.A)
        self.B = nn.Parameter(torch.zeros(d_out, r))     # ★ 零初始化
        self.scaling = alpha / r

    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scaling

base = nn.Linear(64, 32)
lora = MiniLoRA(base)
x = torch.randn(4, 64)
print("B=0 时与基座相等:", torch.allclose(lora(x), base(x), atol=1e-6))
print("可训练参数:", sum(p.numel() for p in lora.parameters() if p.requires_grad),
      "/", sum(p.numel() for p in lora.parameters()))

## 2. 真实注入计划（对着模型看）

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "src.train.lora_utils", "--table"],
                   capture_output=True, text=True, cwd="..")
print(r.stdout or r.stderr)

## 3. 思考题落笔

在下面这个格子写下你的答案（写完就是打卡素材）：

1. 为什么 LoRA 能省显存，而 **QLoRA 还能再省一半**？（提示：基座权重存成什么精度）
2. 客服场景里，一个「看得懂图」的 LoRA，最该更新的是哪些层？

In [ ]:
my_answer = """
1.
2.
"""
print(my_answer)

## 验收清单

- [ ] `--demo` 全部断言通过（**B=0 时 `torch.allclose` 必须为 True**）
- [ ] 能说清 `r` / `alpha` / `dropout` / `target_modules` 各自作用和调参直觉
- [ ] 能说出「为什么客服场景要优先注入 `merger.mlp`」（视觉和语言的对齐层）
- [ ] 知道 `merge_and_unload()` 之后推理为什么和全量模型一样快

**卡住了？** 回看 [`days/day-14.md`](../days/day-14.md) 第五节「容易踩的坑」。

> **明天**：`days/day-15.md` —— 真正开训，第一次看到 loss 下降